In [ ]:
import scanpy as sc
import numpy as np

adata = sc.read_h5ad('/root/autodl-tmp/SVD_degraded_CESC_visium_hd_2um.h5ad')
cdata = sc.read_h5ad('/root/autodl-tmp/SVD_CESC_visium_hd_2um.h5ad')

In [2]:
adata

AnnData object with n_obs × n_vars = 12715913 × 2984
    obs: 'cell_id', 'nucleus_id', 'n_counts'
    var: 'n_cells'
    obsm: 'X_pca_norm', 'spatial'

In [3]:
cdata

AnnData object with n_obs × n_vars = 12715913 × 2984
    obs: 'cell_id', 'nucleus_id', 'n_counts'
    var: 'n_cells'
    obsm: 'X_pca_norm', 'spatial'

In [4]:
import os
import torch
import numpy as np

def generate_sparse_tiles(cdata, adata, save_dir, bin_size=2.0, patch_size=128, overlap=16):
    """
    cdata: Clean Data (Target) - 包含完整细胞掩膜 (cell_id)
    adata: Degraded Data (Input) - 包含细胞核掩膜 (nucleus_id)
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # 1. Coordinate Scaling (假设两者空间坐标已对齐，以 cdata 为基准)
    coords = np.round(cdata.obsm['spatial'] / bin_size).astype(np.int32)
    x_coords, y_coords = coords[:, 0], coords[:, 1]
    W, H = x_coords.max() + 1, y_coords.max() + 1
    
    # 2. Extract 64D PCA Features
    c_features = cdata.obsm['X_pca_norm']  # Target
    a_features = adata.obsm['X_pca_norm']  # Input
    C = 64
    
    # 3. Label Encoding (Masks)
    target_mask_vals = cdata.obs['cell_id'].astype('category').cat.codes.values
    input_mask_vals = adata.obs['nucleus_id'].astype('category').cat.codes.values
    
    # 4. Calculate Offsets for Centered Tiling
    stride = patch_size - overlap
    nx = (W - patch_size) // stride + 1
    ny = (H - patch_size) // stride + 1
    
    rem_w = W - (patch_size + (nx - 1) * stride)
    rem_h = H - (patch_size + (ny - 1) * stride)
    offset_x = rem_w // 2
    offset_y = rem_h // 2
    
    # 5. Sparse Tiling
    valid_count = 0
    channel_idx = np.arange(C)
    
    for y0 in range(offset_y, offset_y + ny * stride, stride):
        for x0 in range(offset_x, offset_x + nx * stride, stride):
            
            # --- Target Expression (64D, from cdata) ---
            c_idx = (y_coords >= y0) & (y_coords < y0 + patch_size) & (x_coords >= x0) & (x_coords < x0 + patch_size)
            if not c_idx.any():
                continue
                
            local_y_c = y_coords[c_idx] - y0
            local_x_c = x_coords[c_idx] - x0
            n_c = len(local_y_c)
            
            ty_c = np.repeat(local_y_c, C)
            tx_c = np.repeat(local_x_c, C)
            tc_c = np.tile(channel_idx, n_c)
            tv_c = c_features[c_idx].flatten()
            
            target_expr = torch.sparse_coo_tensor(
                indices=torch.tensor(np.vstack([ty_c, tx_c, tc_c]), dtype=torch.int64),
                values=torch.tensor(tv_c, dtype=torch.float32),
                size=(patch_size, patch_size, C)
            ).coalesce()
            
            # --- Input Expression (64D, from adata) ---
            a_idx = (y_coords >= y0) & (y_coords < y0 + patch_size) & (x_coords >= x0) & (x_coords < x0 + patch_size)
            local_y_a = y_coords[a_idx] - y0
            local_x_a = x_coords[a_idx] - x0
            n_a = len(local_y_a)
            
            ty_a = np.repeat(local_y_a, C)
            tx_a = np.repeat(local_x_a, C)
            tc_a = np.tile(channel_idx, n_a)
            tv_a = a_features[a_idx].flatten()
            
            input_expr = torch.sparse_coo_tensor(
                indices=torch.tensor(np.vstack([ty_a, tx_a, tc_a]), dtype=torch.int64),
                values=torch.tensor(tv_a, dtype=torch.float32),
                size=(patch_size, patch_size, C)
            ).coalesce()
            
            # --- Masks ---
            m_idx = (y_coords >= y0) & (y_coords < y0 + patch_size) & (x_coords >= x0) & (x_coords < x0 + patch_size)
            local_y = y_coords[m_idx] - y0
            local_x = x_coords[m_idx] - x0
            
            # Target Mask (from cdata)
            t_vals = target_mask_vals[m_idx]
            t_valid = t_vals > 0
            target_mask = torch.sparse_coo_tensor(
                indices=torch.tensor(np.vstack([local_y[t_valid], local_x[t_valid], np.zeros_like(local_y[t_valid])]), dtype=torch.int64),
                values=torch.tensor(t_vals[t_valid], dtype=torch.int32),
                size=(patch_size, patch_size, 1)
            ).coalesce()
            
            # Input Mask (from adata)
            i_vals = input_mask_vals[m_idx]
            i_valid = i_vals > 0
            input_mask = torch.sparse_coo_tensor(
                indices=torch.tensor(np.vstack([local_y[i_valid], local_x[i_valid], np.zeros_like(local_y[i_valid])]), dtype=torch.int64),
                values=torch.tensor(i_vals[i_valid], dtype=torch.int32),
                size=(patch_size, patch_size, 1)
            ).coalesce()
            
            # Save
            tile_data = {
                "input_expr": input_expr,
                "input_nuclei": input_mask,
                "target_expr": target_expr,
                "target_cell_id": target_mask
            }
            torch.save(tile_data, os.path.join(save_dir, f"tile_{y0}_{x0}.pt"))
            valid_count += 1

    print(f"Processed {valid_count} valid sparse tiles.")

In [ ]:
generate_sparse_tiles(cdata, adata, save_dir='/root/autodl-tmp/CESC', bin_size=2.0, patch_size=128, overlap=16)

/tmp/ipykernel_10728/1679191854.py:57: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  target_expr = torch.sparse_coo_tensor(


Processed 1456 valid sparse tiles.


In [ ]:
import shutil

source_dir = "/root/autodl-tmp/LIHC"
output_path = "/root/autodl-tmp/LIHC"  # 不需要加 .zip 后缀，函数会自动添加

shutil.make_archive(output_path, 'zip', source_dir)